# GLASS Interactive SI Shift

Display interactive latest-observed-count vs future SI-shift plots, with GLASS pathogen-antibiotic pairs highlighted.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "amr").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "amr").exists():
    raise RuntimeError(f"Could not find repository root from {Path.cwd()}")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from amr.pipelines import (
    build_glass_match_table,
    build_novel_prediction_table,
    build_si_analysis_paths,
    load_or_build_interactive_rank,
    load_si_and_clean_glass_list,
    plot_future_si_vs_si_shift,
    plot_latest_si_vs_future_si,
    summarize_pathogen_coverage,
)
from amr.si import plot_interactive_latest_observed_count_vs_future_si_shift
from amr.pipeline_config import cfg, cfg_path, load_pipeline_config


## Parameters

In [3]:
PIPELINE_CONFIG = load_pipeline_config(project_root=PROJECT_ROOT)

OUTPUT_DIR = cfg_path(PIPELINE_CONFIG, "paths.output_dir", PROJECT_ROOT)
PARQUET_OUTPUT_DIR = cfg_path(PIPELINE_CONFIG, "paths.parquet_dir", PROJECT_ROOT)
FIGURE_OUTPUT_DIR = cfg_path(PIPELINE_CONFIG, "paths.figure_dir", PROJECT_ROOT)
PARQUET_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR = PARQUET_OUTPUT_DIR
ASSETS_DIR = cfg_path(PIPELINE_CONFIG, "paths.assets_dir", PROJECT_ROOT)

ANTIBIOTIC_ANALYSIS_LEVEL = cfg(PIPELINE_CONFIG, "glass_interactive.antibiotic_analysis_level")
SI_PATHS = build_si_analysis_paths(PROJECT_ROOT, ANTIBIOTIC_ANALYSIS_LEVEL)
YEARLY_SCREENED_PATH = cfg_path(PIPELINE_CONFIG, "paths.yearly_screened", PROJECT_ROOT)
YEARLY_SI_PATH = SI_PATHS["yearly_si"]
RANK_OUTPUT_PATH = SI_PATHS["rank_output"]

GLASS_XLSX_PATH = cfg_path(PIPELINE_CONFIG, "paths.glass_xlsx", PROJECT_ROOT)
GLASS_CLEANED_PATH = cfg_path(PIPELINE_CONFIG, "paths.glass_cleaned", PROJECT_ROOT)
BACTERIA_SYNONYM_PATH = cfg_path(PIPELINE_CONFIG, "paths.pathogens", PROJECT_ROOT)
ANTIBIOTIC_SYNONYM_PATH = cfg_path(PIPELINE_CONFIG, "paths.antibiotic_synonyms", PROJECT_ROOT)
ANTIBIOTIC_COARSENING_PATH = cfg_path(PIPELINE_CONFIG, "paths.antibiotic_coarsening", PROJECT_ROOT)

# Reuse SI fit window settings so interactive plots stay aligned with notebook 02 outputs.
FIT_START_YEAR = cfg(PIPELINE_CONFIG, "si.fit_start_year")
FIT_END_YEAR = cfg(PIPELINE_CONFIG, "si.fit_end_year")
CURVE_FIT_END_YEAR = cfg(PIPELINE_CONFIG, "si.curve_fit_end_year")
NUM_FUTURE_YEARS = cfg(PIPELINE_CONFIG, "glass_interactive.num_future_years")
LOESS_FRAC = cfg(PIPELINE_CONFIG, "si.loess_frac")
LOESS_DEGREE = cfg(PIPELINE_CONFIG, "si.loess_degree")
TAIL_DAMPING_YEARS = cfg(PIPELINE_CONFIG, "si.tail_damping_years")

# Interactive filtering and plot layout controls.
MIN_LAST_AVAILABLE_COUNT = cfg(PIPELINE_CONFIG, "glass_interactive.min_last_available_count")
MIN_COUNT_ABS_PCT_ERROR_MEAN = cfg(PIPELINE_CONFIG, "glass_interactive.min_count_abs_pct_error_mean")
MAX_COUNT_ABS_PCT_ERROR_MEAN = cfg(PIPELINE_CONFIG, "glass_interactive.max_count_abs_pct_error_mean")
MIN_COUNTS_BACKGROUND = cfg(PIPELINE_CONFIG, "glass_interactive.min_counts_background")
MIN_ABS_DELTA_SI = cfg(PIPELINE_CONFIG, "glass_interactive.min_abs_delta_si")
MIN_SI_DELTA = cfg(PIPELINE_CONFIG, "glass_interactive.min_si_delta")
DISPLAY_NAME = cfg(PIPELINE_CONFIG, "glass_interactive.display_name")
MAX_TEXT_LABELS = cfg(PIPELINE_CONFIG, "glass_interactive.max_text_labels")
WIDTH = cfg(PIPELINE_CONFIG, "glass_interactive.width")
HEIGHT = cfg(PIPELINE_CONFIG, "glass_interactive.height")

if not YEARLY_SCREENED_PATH.exists():
    raise FileNotFoundError(f"Missing {YEARLY_SCREENED_PATH}. Run notebook 01 to create screened AMR pairs first.")
if not YEARLY_SI_PATH.exists():
    raise FileNotFoundError(f"Missing {YEARLY_SI_PATH}. Run notebook 02 after notebook 01 to build SI outputs.")
if YEARLY_SI_PATH.stat().st_mtime < YEARLY_SCREENED_PATH.stat().st_mtime:
    raise RuntimeError(
        f"{YEARLY_SI_PATH} is older than {YEARLY_SCREENED_PATH}. "
        "Run notebook 02 after the latest notebook 01 LLM/screened-pair run."
    )


## Load SI And Clean GLASS List

In [4]:
si_df, glass_pairs_df = load_si_and_clean_glass_list(
    YEARLY_SI_PATH,
    GLASS_XLSX_PATH,
    bacteria_synonym_path=BACTERIA_SYNONYM_PATH,
    antibiotic_synonym_path=ANTIBIOTIC_SYNONYM_PATH,
    antibiotic_coarsening_path=ANTIBIOTIC_COARSENING_PATH,
    glass_cleaned_path=GLASS_CLEANED_PATH,
)

si_source_df = pd.read_parquet(YEARLY_SCREENED_PATH)
si_pathogen_summary_df = summarize_pathogen_coverage(
    source_df=si_source_df,
    modeled_df=si_df,
    source_label="screened AMR input from notebook 01",
    modeled_label="filtered SI dataset from notebook 02",
)
display(si_pathogen_summary_df.query("status != 'present_in_both'"))
display(glass_pairs_df.head(20))

Loaded SI rows: 6,611 from /run/media/fabrizio/06bb7271-2161-43a4-91f1-98f9b67e9ab2/home/fabrizio/code/Antimicrobial-Resistance/outputs/parquets/yearly_si.parquet
Pathogens: 25; antibiotics/levels: 69
Saved cleaned GLASS rows: 89 to /run/media/fabrizio/06bb7271-2161-43a4-91f1-98f9b67e9ab2/home/fabrizio/code/Antimicrobial-Resistance/assets/glass_amr_cleaned.csv
screened AMR input from notebook 01 pathogens: 48
filtered SI dataset from notebook 02 pathogens: 25
Pathogens present in screened AMR input from notebook 01 but absent from filtered SI dataset from notebook 02: 23
Aggregatibacter actinomycetemcomitans, Bacillus anthracis, Bordetella pertussis, Borrelia burgdorferi, Brucella melitensis, Burkholderia cepacia, Burkholderia pseudomallei, Chlamydia pneumoniae, Chlamydia trachomatis, Coxiella burnetii, Francisella tularensis, Legionella pneumophila, Moraxella catarrhalis, Mycobacterium abscessus, Mycobacterium avium, Mycobacterium leprae, Mycoplasmoides pneumoniae, Neisseria meningiti

,pathogen,source_rows,source_publications,source_pairs,modeled_rows,modeled_publications,modeled_pairs,in_source,in_modeled,status
0,Aggregatibacter actinomycetemcomitans,60,36,15,0.0,0.0,0.0,True,False,missing_from_modeled
1,Bacillus anthracis,41,31,30,0.0,0.0,0.0,True,False,missing_from_modeled
2,Bordetella pertussis,30,20,11,0.0,0.0,0.0,True,False,missing_from_modeled
3,Borrelia burgdorferi,21,18,14,0.0,0.0,0.0,True,False,missing_from_modeled
4,Brucella melitensis,68,48,24,0.0,0.0,0.0,True,False,missing_from_modeled
5,Burkholderia cepacia,123,89,38,0.0,0.0,0.0,True,False,missing_from_modeled
6,Burkholderia pseudomallei,91,90,30,0.0,0.0,0.0,True,False,missing_from_modeled
7,Chlamydia pneumoniae,28,22,20,0.0,0.0,0.0,True,False,missing_from_modeled
8,Chlamydia trachomatis,75,54,31,0.0,0.0,0.0,True,False,missing_from_modeled
9,Coxiella burnetii,6,3,4,0.0,0.0,0.0,True,False,missing_from_modeled


,raw_pathogen,raw_antibiotic,pathogen,antibiotic,pathogen_standardized,antibiotic_standardized,Class,Simplified categories,High level categories,Origin
0,Acinetobacter spp.,amikacin,Acinetobacter baumannii,Amikacin,True,True,Aminoglycosides,Aminocyclitols/Aminoglycosides,Aminocyclitols/Aminoglycosides,Naturally occurring/modified
1,Acinetobacter spp.,Colistin,Acinetobacter baumannii,Colistin,True,False,Polymyxins,Miscellaneous,Other,Naturally occurring/modified
2,Acinetobacter spp.,doripenem,Acinetobacter baumannii,Doripenem,True,True,Carbapenems,Penems/Carbapenems,Beta lactam and related,Naturally occurring/modified
3,Acinetobacter spp.,Gentamicin,Acinetobacter baumannii,Gentamicin,True,False,Aminoglycosides,Aminocyclitols/Aminoglycosides,Aminocyclitols/Aminoglycosides,Naturally occurring/modified
4,Acinetobacter spp.,Imipenem,Acinetobacter baumannii,Imipenem/cilastatin,True,True,Carbapenems,Penems/Carbapenems,Beta lactam and related,Naturally occurring/modified
5,Acinetobacter spp.,meropenem,Acinetobacter baumannii,Meropenem,True,True,Carbapenems,Penems/Carbapenems,Beta lactam and related,Naturally occurring/modified
6,Acinetobacter spp.,minocycline,Acinetobacter baumannii,Minocycline,True,True,Tetracyclines,Tetracyclines and related,Tetracyclines and related,Naturally occurring/modified
7,Acinetobacter spp.,Tigecycline,Acinetobacter baumannii,Tigecycline,True,False,Glycylcyclines,Tetracyclines and related,Tetracyclines and related,Naturally occurring/modified
8,E. coli,Cefepime,Escherichia coli,Cefepime,True,False,Fourth-generation-cephalosporins,Cephalosporins,Beta lactam and related,Naturally occurring/modified
9,E. coli,cefotaxime,Escherichia coli,Cefotaxime,True,True,Third-generation-cephalosporins,Cephalosporins,Beta lactam and related,Naturally occurring/modified


## Build Or Load Rank Data

In [5]:
rank_df = load_or_build_interactive_rank(
    si_df,
    RANK_OUTPUT_PATH,
    source_path=YEARLY_SI_PATH,
    fit_start_year=FIT_START_YEAR,
    fit_end_year=FIT_END_YEAR,
    curve_fit_end_year=CURVE_FIT_END_YEAR,
    num_future_years=NUM_FUTURE_YEARS,
    loess_frac=LOESS_FRAC,
    loess_degree=LOESS_DEGREE,
    tail_damping_years=TAIL_DAMPING_YEARS,
    verbose=True,
)
display(rank_df)

Recomputing rank data because cached fit metadata does not match the requested endpoints.
[1/363] ranking Acinetobacter baumannii vs Amikacin
[2/363] ranking Acinetobacter baumannii vs Ampicillin
[3/363] ranking Acinetobacter baumannii vs Ampicillin/sulbactam
[4/363] ranking Acinetobacter baumannii vs Aztreonam
[5/363] ranking Acinetobacter baumannii vs Cefepime
[6/363] ranking Acinetobacter baumannii vs Cefoperazone
[7/363] ranking Acinetobacter baumannii vs Cefotaxime
[8/363] ranking Acinetobacter baumannii vs Ceftazidime
[9/363] ranking Acinetobacter baumannii vs Ceftolozane/tazobactam
[10/363] ranking Acinetobacter baumannii vs Ceftriaxone
[11/363] ranking Acinetobacter baumannii vs Chloramphenicol
[12/363] ranking Acinetobacter baumannii vs Ciprofloxacin
[13/363] ranking Acinetobacter baumannii vs Colistin
[14/363] ranking Acinetobacter baumannii vs Gentamicin
[15/363] ranking Acinetobacter baumannii vs Levofloxacin
[16/363] ranking Acinetobacter baumannii vs Meropenem
[17/363] ra

,pathogen,antibiotic,reference_year,future_year,num_future_years,fit_start_year,fit_end_year,curve_fit_end_year,post_fit_validation_end_year,raw_latest_available_year,...,tail_damping_years,validation_loess_frac,validation_loess_degree,validation_tail_damping_years,status,error,current_rank,future_rank,rank_change,abs_rank_change
0,Acinetobacter baumannii,Trimethoprim,2025,2028,3,2000,2023,2025,2022,2022,...,None,0.6,1,None,ok,None,123.0,86.0,37.0,37.0
1,Acinetobacter baumannii,Sulfamethoxazole/trimethoprim,2025,2028,3,2000,2023,2025,2022,2022,...,None,0.6,1,None,ok,None,111.0,75.0,36.0,36.0
2,Acinetobacter baumannii,Ceftriaxone,2025,2028,3,2000,2023,2025,2022,2022,...,None,0.6,1,None,ok,None,85.0,50.0,35.0,35.0
3,Klebsiella pneumoniae,Colistin,2025,2028,3,2000,2023,2025,2022,2022,...,None,0.6,1,None,ok,None,240.0,274.0,-34.0,34.0
4,Klebsiella pneumoniae,Fosfomycin,2025,2028,3,2000,2023,2025,2022,2022,...,None,0.6,1,None,ok,None,130.0,163.0,-33.0,33.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
358,Escherichia coli,Colistin,2025,2028,3,2000,2023,2025,2022,2022,...,None,0.6,1,None,ok,None,354.0,354.0,0.0,0.0
359,Escherichia coli,Ciprofloxacin,2025,2028,3,2000,2023,2025,2022,2022,...,None,0.6,1,None,ok,None,360.0,360.0,0.0,0.0
360,Escherichia coli,Ampicillin,2025,2028,3,2000,2023,2025,2022,2022,...,None,0.6,1,None,ok,None,361.0,361.0,0.0,0.0
361,Klebsiella pneumoniae,Meticillin,2025,2028,3,2000,2023,2025,2022,2021,...,None,0.6,1,None,ok,None,362.0,362.0,0.0,0.0


## Interactive Latest Count Vs Future SI Shift

In [6]:
latest_count_shift_result = plot_interactive_latest_observed_count_vs_future_si_shift(
    rank_df,
    glass_pairs_df=glass_pairs_df,
    min_last_available_count=MIN_LAST_AVAILABLE_COUNT,
    min_count_abs_pct_error_mean=MIN_COUNT_ABS_PCT_ERROR_MEAN,
    max_count_abs_pct_error_mean=MAX_COUNT_ABS_PCT_ERROR_MEAN,
    min_counts_background=MIN_COUNTS_BACKGROUND,
    min_abs_delta_si=MIN_ABS_DELTA_SI,
    min_si_delta=MIN_SI_DELTA,
    display_name=DISPLAY_NAME,
    max_text_labels=MAX_TEXT_LABELS,
    width=WIDTH,
    height=HEIGHT,
    pdf_path=FIGURE_OUTPUT_DIR / "interactive_latest_observed_count_vs_future_si_shift.pdf",
    show=True,
)
latest_count_shift_df = latest_count_shift_result["plot_df"]
print(f"Displayed rows: {len(latest_count_shift_df):,}; GLASS matches: {int(latest_count_shift_df['is_glass_pair'].sum()):,}")
if latest_count_shift_result["pdf_path"]:
    print(f"Saved PDF: {latest_count_shift_result['pdf_path']}")
elif latest_count_shift_result.get("pdf_error"):
    print(f"PDF export failed: {latest_count_shift_result['pdf_error']}")

Displayed rows: 26; GLASS matches: 10
Saved PDF: /run/media/fabrizio/06bb7271-2161-43a4-91f1-98f9b67e9ab2/home/fabrizio/code/Antimicrobial-Resistance/outputs/figures/interactive_latest_observed_count_vs_future_si_shift.pdf


## Display GLASS Matches In Plot

In [7]:
glass_plot_matches_df = build_glass_match_table(latest_count_shift_df)
display(glass_plot_matches_df)

,pathogen,antibiotic,latest_available_year,future_year,latest_available_count,future_predicted_count,latest_smoothed_SI,future_predicted_SI,future_minus_latest_smoothed_SI,post_fit_count_abs_pct_error_mean,glass_group_column
0,Haemophilus influenzae,Amoxicillin/clavulanic-acid,2025,2028,35.765241,79.071309,1.416992,1.780380,0.363388,NaN,antibiotic
1,Neisseria gonorrhoeae,Gentamicin,2025,2028,35.743742,133.685511,0.595812,1.146887,0.551075,NaN,antibiotic
2,Salmonella enterica,Azithromycin,2025,2028,28.615929,62.280426,0.887815,1.249220,0.361404,NaN,antibiotic
3,Salmonella enterica,Ciprofloxacin,2025,2028,13.709395,12.624812,0.263835,0.231876,-0.031958,NaN,antibiotic
4,Salmonella enterica,Ceftriaxone,2025,2028,13.468810,16.927589,0.428665,0.527486,0.098821,NaN,antibiotic
5,Neisseria gonorrhoeae,Azithromycin,2025,2028,13.259541,13.129397,0.451755,0.453622,0.001867,NaN,antibiotic
6,Pseudomonas aeruginosa,Meropenem,2025,2028,12.097249,11.110713,0.316781,0.281610,-0.035171,NaN,antibiotic
7,Neisseria gonorrhoeae,Ciprofloxacin,2025,2028,11.220686,10.654272,0.088663,0.056086,-0.032578,NaN,antibiotic
8,Staphylococcus aureus,Oxacillin,2025,2028,10.360907,8.457221,-0.099513,-0.198651,-0.099138,NaN,antibiotic
9,Neisseria gonorrhoeae,Ceftriaxone,2025,2028,10.019381,8.205560,0.215267,0.129379,-0.085888,NaN,antibiotic


## Novel Non-GLASS Predictions

These are plotted pairs that do not appear in the cleaned GLASS list, sorted as candidate novel predictions of interest.

In [8]:
novel_prediction_df = build_novel_prediction_table(latest_count_shift_df)
print(f"Novel non-GLASS predictions displayed: {len(novel_prediction_df):,}")
display(novel_prediction_df)

Novel non-GLASS predictions displayed: 16


,pathogen,antibiotic,latest_available_year,future_year,latest_available_count,future_predicted_count,latest_smoothed_SI,future_predicted_SI,future_minus_latest_smoothed_SI,post_fit_count_abs_pct_error_mean
0,Pseudomonas aeruginosa,Ceftazidime/avibactam,2025,2028,246.641689,3203.104644,1.899637,3.018909,1.119272,NaN
1,Vibrio parahaemolyticus,Ampicillin,2025,2028,110.979651,761.061290,1.442320,2.280542,0.838222,NaN
2,Acinetobacter baumannii,Sulfamethoxazole,2025,2028,10.653246,29.315248,0.345668,0.740427,0.394759,NaN
3,Listeria monocytogenes,Gentamicin,2025,2028,11.326684,27.667900,0.619991,0.981440,0.361448,NaN
4,Salmonella enterica,Colistin,2025,2028,19.940326,43.978310,0.575838,0.907361,0.331524,NaN
5,Klebsiella pneumoniae,Polymyxin-B,2025,2028,10.663731,20.496061,0.450608,0.680189,0.229581,NaN
6,Pseudomonas aeruginosa,Ceftolozane/tazobactam,2025,2028,10.797838,14.811837,0.430267,0.537094,0.106826,NaN
7,Staphylococcus aureus,Cefoxitin,2025,2028,10.992784,12.367180,0.407507,0.446130,0.038623,NaN
8,Staphylococcus aureus,Clindamycin,2025,2028,17.857992,17.665044,0.283232,0.268916,-0.014317,NaN
9,Staphylococcus aureus,Ampicillin,2025,2028,12.159376,11.946672,0.141965,0.126428,-0.015536,NaN


## Future SI Vs Future SI Shift

This uses the same y-axis definition as the latest-count plot: predicted future SI minus smoothed SI at the latest available year.

In [ ]:
future_si_shift_result = plot_future_si_vs_si_shift(
    latest_count_shift_df,
    min_count_abs_pct_error_mean=MIN_COUNT_ABS_PCT_ERROR_MEAN,
    max_count_abs_pct_error_mean=MAX_COUNT_ABS_PCT_ERROR_MEAN,
    display_name=DISPLAY_NAME,
    max_text_labels=MAX_TEXT_LABELS,
    width=WIDTH,
    height=HEIGHT,
    pdf_path=FIGURE_OUTPUT_DIR / "future_si_vs_si_shift.pdf",
    show=True,
)
future_si_shift_fig = future_si_shift_result["figure"]
if future_si_shift_result["pdf_path"]:
    print(f"Saved PDF: {future_si_shift_result['pdf_path']}")
elif future_si_shift_result.get("pdf_error"):
    print(f"PDF export failed: {future_si_shift_result['pdf_error']}")

Saved PDF: /run/media/fabrizio/06bb7271-2161-43a4-91f1-98f9b67e9ab2/home/fabrizio/code/Antimicrobial-Resistance/outputs/figures/future_si_vs_si_shift.pdf


: 